<h1 style="color: #4166c4; font-weight: 700;">Preprocessing</h1>

## Table of Contents

- [1. Introduction](#introduction)
- [2. Metadata preprocessing](#metadata-preprocessing)
  - [2.1 Correcting inconsistencies](#correcting-inconsistencies)
    - [2.1.1 Correcting artist names](#artist-names)
      - [2.1.1.1 Infer artist names from title structure](#title-inference)
      - [2.1.1.2 Review artist labels containing `Genius`](#genius-review)
    - [2.1.2 Correcting years](#release-years)
  - [2.2 Lyrics normalization](#lyrics-normalization)
  - [2.3 Duplicate-lyrics audit](#duplicate-audit)

<h2 style="color: #4166c4; font-weight: 400;">1. Introduction <a id="introduction"></a></h2>


This notebook independently prepares the supplied training and test datasets by reviewing and correcting likely artist-name and year inconsistencies, then normalizing lyric text. It locates the project root automatically, so the data and helper imports work whether the notebook is opened from the project root or `Notebooks/`. The cells load `data/train.csv`, `data/test.csv`, and `data/sample_submission.csv`; the corrected frames remain available as `X_train` and `X_test` during this notebook session.

Artist and year corrections use Genius lookups for unresolved candidates. Those lookups may take time or fail when data is unavailable; the code reports unresolved entries rather than inventing values. Keep the original CSV files unchanged and save corrected outputs explicitly if you need persistent copies.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'data').is_dir() and (path / 'tools').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root containing data/ and tools/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
df_train = pd.read_csv(PROJECT_ROOT / 'data' / 'train.csv')

In [2]:
from sklearn.model_selection import train_test_split
import importlib
import tools.utils as utils
utils = importlib.reload(utils)
get_info_batch = utils.get_info_batch
import tools.preprocessing as preprocessing
preprocessing = importlib.reload(preprocessing)
fix_all_translation_artists = preprocessing.fix_all_translation_artists
correct_pre_years = preprocessing.correct_pre_years
remove_duplicated_lyrics = preprocessing.remove_duplicated_lyrics


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\almad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\almad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\almad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
X_train = df_train.drop(columns=['tag'])
y_train = df_train['tag']

In [4]:
X_test=pd.read_csv(PROJECT_ROOT / 'data' / 'test.csv')
y_test=pd.read_csv(PROJECT_ROOT / 'data' / 'sample_submission.csv')

<h2 style="color: #4166c4; font-weight: 400;">2. Metadata preprocessing <a id="metadata-preprocessing"></a></h2>

<h3 style="color: #597edb; font-weight: 400;">2.1 Correcting inconsistencies <a id="correcting-inconsistencies"></a></h3>

<h4>2.1.1 Correcting artist names <a id="artist-names"></a></h4>

Some records list a Genius translation, transliteration, or romanization account as the artist. The following steps identify likely platform-account labels in the training and test frames, first infer an artist from a title when its format allows, and then query Genius for remaining candidates. Unresolved names are left unchanged for review; a match on the word `Genius` alone is not sufficient evidence that an artist label is wrong.

Identify likely Genius translation or romanization account names and inspect their titles. These are candidates for correction, not confirmed errors; review the displayed examples before relying on inferred names.

In [5]:
translation_account_pattern = (
    r'(?i)\bgenius\b.*\b(?:translat\w*|trad\w*|romaniz\w*|romanis\w*|transliter\w*)\b')

In [6]:
train_translation_mask = X_train['artist'].str.contains(
    translation_account_pattern, na=False, regex=True
) #checking which rows have artists that are translation accounts in train
test_translation_mask = X_test['artist'].str.contains(
    translation_account_pattern, na=False, regex=True
) #checking which rows have artists that are translation accounts in test

#extracting this specififc info about those rows so we can visualize them and chek if there's a pattern
train_miss_artists = X_train.loc[
    train_translation_mask, ['id', 'title', 'artist']
] 
test_miss_artists = X_test.loc[
    test_translation_mask, ['id', 'title', 'artist']
]


print(f"Train: {len(train_miss_artists)} rows")
display(train_miss_artists['artist'].value_counts().rename_axis('artist').reset_index(name='count'))
display(train_miss_artists.head(10))

print(f"Test: {len(test_miss_artists)} rows")
display(test_miss_artists['artist'].value_counts().rename_axis('artist').reset_index(name='count'))
display(test_miss_artists.head(10))

Train: 606 rows


,artist,count
0,Genius English Translations,555
1,Genius Traductions Franaises,21
2,Genius Romanizations,13
3,Genius Brasil Tradues,9
4,Genius South Africa Translations,3
5,Genius Traduzioni Italiane,2
6,Genius Farsi Translations ( ),1
7,Genius Swedish Translations,1
8,Genius Translations,1


,id,title,artist
116,6815917,Ferbinho - Lionel English Translation,Genius English Translations
449,7634567,Def. - WANT U English Translation,Genius English Translations
459,7550031,Sogumm 소금 Dress - Honey bee ft. PENOMECO 페노메코 ...,Genius English Translations
526,7749379,Ricky Martin - Náufrago English Translation,Genius English Translations
617,3506177,Drake - Legend Tradução em Português,Genius Brasil Tradues
926,7366990,Coogie - Swipe Feat. Ourealgoat Touch the Sky...,Genius English Translations
947,4477095,Chanyeol EXO - 봄 여름 가을 겨울 SSFW English Transla...,Genius English Translations
1373,6706099,Orkundk - Marsa Gönder Beni Elon Musk English ...,Genius English Translations
1835,5679491,Eraserheads - Minsan English Translation,Genius English Translations
2228,6564362,Damons Year - August,Genius English Translations


Test: 122 rows


,artist,count
0,Genius English Translations,117
1,Genius Brasil Tradues,2
2,Genius Swedish Translations,1
3,Genius Romanizations,1
4,Genius Traductions Franaises,1


,id,title,artist
406,7349077,Hov1 - När jag ser dig English Translation,Genius Swedish Translations
814,4691789,Гречка Grechka – Люби меня люби Love Me Love M...,Genius English Translations
847,5449591,Nu Drama - Ikigai English Translation,Genius English Translations
930,7667325,中川 翔子 Shoko Nakagawa - 空色デイズ Sorairo Days Engl...,Genius English Translations
1207,7519904,10CM - 서랍 Drawer English Translation,Genius English Translations
2635,5573039,Mahmood - Moonlight popolare ft. Massimo Peric...,Genius English Translations
3013,7252584,APRIL - Oh my mistake English Translation,Genius English Translations
3544,5614170,Anuel AA - Nubes Negras English Translation,Genius English Translations
3898,7602975,Moon Byul - 너만 들었으면 좋겠다 For Me English Transla...,Genius English Translations
4661,498735,Drake - Karaoke Tradução em Português,Genius Brasil Tradues


<h5>Infer artist names from title structure <a id="title-inference"></a></h5>

In [7]:

X_train = fix_all_translation_artists(X_train)
X_test = fix_all_translation_artists(X_test)

#checking how many rows couldn't be fixed by the structure approach
train_remaining_translation_mask = X_train['artist'].str.contains(
    translation_account_pattern, na=False, regex=True
)
test_remaining_translation_mask = X_test['artist'].str.contains(
    translation_account_pattern, na=False, regex=True
)

print(f"Train remaining translation-account rows: {train_remaining_translation_mask.sum()}")
display(X_train.loc[train_remaining_translation_mask, 'artist'].value_counts().rename_axis('artist').reset_index(name='count'))

print(f"Test remaining translation-account rows: {test_remaining_translation_mask.sum()}")
display(X_test.loc[test_remaining_translation_mask, 'artist'].value_counts().rename_axis('artist').reset_index(name='count'))


Train remaining translation-account rows: 16


,artist,count
0,Genius English Translations,15
1,Genius South Africa Translations,1


Test remaining translation-account rows: 5


,artist,count
0,Genius English Translations,5


In [8]:
#getting those rows that couldnt be corrected by the structure
for name, data in [('Train', X_train), ('Test', X_test)]:
    remaining_mask = data['artist'].str.contains(
        translation_account_pattern, na=False, regex=True
    )
    remaining_ids = data.loc[remaining_mask, 'id']

    print(f"{name}: querying {len(remaining_ids)} remaining translation-account rows...")
    if not remaining_ids.empty:
        fetched = get_info_batch(remaining_ids, artist=True) # trying to correct those artists trough the Genius API
        data.loc[remaining_mask, 'artist'] = (
            data.loc[remaining_mask, 'id'].map(fetched['artist'])
            .where(lambda artists: artists.notna(), data.loc[remaining_mask, 'artist'])
        )
 #checking how many rows couldn't be fixed by the API approach
    still_remaining_mask = data['artist'].str.contains(
        translation_account_pattern, na=False, regex=True
    )
    still_remaining_rows = data.loc[
        still_remaining_mask, ['id', 'title', 'artist']
    ]
    print(f"{name}: remaining translation-account rows after lookup: {len(still_remaining_rows)}")
    display(still_remaining_rows)

Train: querying 16 remaining translation-account rows...
Train: remaining translation-account rows after lookup: 5


,id,title,artist
72559,7388735,Dangote Lyrics English Translation,Genius English Translations
86688,5952033,LLuvia en el Paraiso English Translation,Genius English Translations
111692,3953910,RAINING Korean Ver. English Translation,Genius English Translations
121008,4413755,So What English Translation,Genius English Translations
123064,6963138,Sicilian Panda RMX,Genius English Translations


Test: querying 5 remaining translation-account rows...
Test: remaining translation-account rows after lookup: 1


,id,title,artist
10498,3248493,B.O.C English Translation,Genius English Translations


`fix_all_translation_artists` extracts an artist from titles that follow a recognizable `Artist - Title` pattern. The following lookup attempts to resolve remaining candidates through Genius. Any candidate the title rule or lookup cannot resolve stays unchanged, and the output shows those cases for review.

This check lists artist labels containing `Genius` across both splits. It is a review aid only: legitimate artist names can contain the same word, so do not treat every match as an incorrect platform credit.

<h5> Review artist labels containing `Genius` <a id="genius-review"></a></h5>

In [9]:
# Show all artist values that contain the word "Genius" (case-insensitive)
genius_accounts = (
    pd.concat([X_train['artist'], X_test['artist']])
    .dropna()
    .loc[lambda s: s.str.contains('genius', case=False)]
    .value_counts()
)
print(genius_accounts.to_string())
print(f'total sum: {genius_accounts.sum()}')


artist
Genius                                13
Outside the Lines With Rap Genius     13
Genius Users                          13
Polskie tumaczenia Genius             12
Sports Genius                          6
Genius English Translations            6
Perfume Genius                         5
Rock Genius                            5
Fashion Genius                         5
Rap Genius                             4
Dotard Genius                          4
Game Genius                            3
Screen Genius                          3
Genius Dream Album                     3
Lit Genius                             3
Eugenius (band)                        3
Genius Community                       2
Genius Editors                         2
Genius UK                              2
Genius Lists                           2
DJ Muggs & GZA The Genius              2
Math Genius                            2
News Genius                            2
Genius South Africa                    2
MisUnders

Inspect the values listed above and correct only labels supported by reliable evidence. This notebook leaves ambiguous names unchanged rather than making additional API requests or guessing.

<h4>2.1.2 Correcting years <a id="release-years"></a></h4>

`correct_pre_years` checks records with a year earlier than 1900 and attempts to retrieve a release date from Genius. A returned year of 1800 or later replaces the current value. Earlier API years are classified as suspicious, and records with no usable API year are classified as not found; in both cases the year is set to missing and the song IDs are retained for review. The function returns updated frames, which are assigned back to `X_train` and `X_test`.

In [10]:
X_train, train_years_resolved_ids, train_years_suspicious_ids, train_years_not_found_ids = correct_pre_years(
    1900, X_train, 'Train'
)
X_test, test_years_resolved_ids, test_years_suspicious_ids, test_years_not_found_ids = correct_pre_years(
    1900, X_test, 'Test'
)

Train: 653 rows with years before 1900


NameError: name 'get_info_batch' is not defined

In [ ]:
def summarize_years(frame, ids, split, status):
    years = pd.to_numeric(
        frame.loc[frame['id'].isin(ids), 'year'],
        errors='coerce'
    ).dropna()

    return {
        'split': split,
        'status': status,
        'rows': len(ids),
        'rows_with_year': len(years),
        'minimum_year': years.min() if not years.empty else None,
        'years_below_1900': int((years < 1900).sum()),
        'maximum_year': years.max() if not years.empty else None,
    }




year_summary = pd.DataFrame([
    summarize_years(X_train, train_years_resolved_ids, 'Train', 'Resolved'),
    summarize_years(X_test, test_years_resolved_ids, 'Test', 'Resolved'),
    summarize_years(X_train, train_years_suspicious_ids, 'Train', 'Suspicious'),
    summarize_years(X_test, test_years_suspicious_ids, 'Test', 'Suspicious'),
    summarize_years(X_train, train_years_not_found_ids, 'Train', 'Not found'),
    summarize_years(X_test, test_years_not_found_ids, 'Test', 'Not found'),
])

display(year_summary)

The summary below reports the lookup results for the current data and separates resolved, suspicious, and not-found IDs. Counts can change if the source data or API responses change. Missing years are not automatically imputed here. We will later impute them using artist-level statistics.

<h3 style="color: #597edb; font-weight: 400;">2.2 Lyrics normalization <a id="lyrics-normalization"></a></h3>

Normalize the `lyrics` text in each frame before modeling while preserving accents, non-Latin scripts, emoji, and meaningful punctuation. The project helper `normalize_lyrics_inplace` in `tools/preprocessing.py` applies `normalize_lyrics` from `tools/text_quality.py` to the `lyrics` column and updates the supplied DataFrame in place. Use it on both splits:

```python
from tools.preprocessing import normalize_lyrics_inplace

normalize_lyrics_inplace(X_train)
normalize_lyrics_inplace(X_test)
```

This is conservative text normalization, not task-specific tokenization, stopword removal, or stemming. Keep the original source files unchanged and compare any more aggressive transformations separately for each modeling objective.

In [ ]:
from tools.text_quality import normalize_lyrics

X_train['lyrics'] = X_train['lyrics'].apply(
        lambda text: normalize_lyrics(text) if isinstance(text, str) else text
    )
X_test['lyrics'] = X_test['lyrics'].apply(
        lambda text: normalize_lyrics(text) if isinstance(text, str) else text
    )

Before saving, `remove_duplicated_lyrics` keeps one training row per repeated lyric, preferring titles without remix, cover, instrumental, live, and similar version cues. If candidates remain tied, it prefers the earliest known year and then the first row. These metadata clues are only a best-effort way to retain the likely original recording. It also removes training lyrics that occur in test, while leaving both test features and submission targets unchanged.

In [ ]:
train_rows_before = len(X_train)
X_train, y_train, X_test, y_test = remove_duplicated_lyrics(
    X_train, y_train, X_test, y_test
)
print(f"Removed {train_rows_before - len(X_train)} duplicate-lyrics rows from train.")
print(f"Test remains unchanged at {len(X_test)} rows.")

In [ ]:
X_train.to_csv(PROJECT_ROOT / 'data' / 'normalized_x_train.csv', index=False)
X_test.to_csv(PROJECT_ROOT / 'data' / 'normalized_x_test.csv', index=False)

<h3 style="color: #597edb; font-weight: 400;">2.3 Duplicate-lyrics identification and removal <a id="duplicate-audit"></a></h3>



This section checks for repeated lyrics within each split and for exact lyric matches between the training and test sets. The first table is the audit of the loaded files before the cleanup below. Missing and whitespace-only lyrics are excluded. The cleanup keeps one training row per repeated lyric, favoring titles without version cues (such as remix, cover, or live) and then the earliest known year. It removes training rows whose lyrics also occur in test, while leaving the test set unchanged.

In [11]:
from tools.utils import checking_duplicates

normalized_x_train = pd.read_csv(PROJECT_ROOT / 'data' / 'normalized_x_train.csv')
normalized_x_test = pd.read_csv(PROJECT_ROOT / 'data' / 'normalized_x_test.csv')

duplicate_report = checking_duplicates(normalized_x_train, normalized_x_test)
display(duplicate_report['summary'])

,dataset,rows_with_lyrics,unique_duplicate_lyrics,duplicate_rows
0,dataset1,134967.0,187,388
1,dataset2,33742.0,11,24
2,dataset1 vs dataset2,NaN,83,130


The first two rows report duplicates within each split: `dataset1` is the training set and `dataset2` is the test set. The training set has 187 distinct repeated lyric texts across 388 rows; the test set has 11 distinct repeated lyric texts across 24 rows. The comparison row reports 83 distinct lyric texts shared between train and test, producing 130 matching row pairs. These cross-split matches are removed from training to prevent lyric leakage; test rows are retained.

In [12]:
normalized_x_train, y_train, normalized_x_test, y_test = remove_duplicated_lyrics(
    normalized_x_train.copy(),
    y_train.copy(),
    normalized_x_test.copy(),
    y_test.copy(),
)

for name, frame in [
    ("Train", normalized_x_train),
    ("Test", normalized_x_test),
]:
    duplicate_mask = (
        frame["lyrics"].notna()
        & frame["lyrics"].duplicated(keep=False)
    )
    print(f"{name}: {duplicate_mask.sum()} rows with duplicated lyrics")
    display(frame.loc[duplicate_mask].sort_values("lyrics"))

shared_lyrics = set(normalized_x_train["lyrics"].dropna()) & set(
    normalized_x_test["lyrics"].dropna()
)
print(f"Shared lyrics between train and test: {len(shared_lyrics)}")

Train: 0 rows with duplicated lyrics


,title,artist,year,views,features,lyrics,id


Test: 24 rows with duplicated lyrics


,title,artist,year,views,features,lyrics,id
57,Piotr Czerski’s “We the Web Kids” Liza Paley,#CA346,2014.0,21,"{""Tony Tran""}",1. We grew up with the Internet and on the Int...,559911
29409,Piotr Czerski’s “We the Web Kids” 21,Jerdotcom,2014.0,0,{},1. We grew up with the Internet and on the Int...,585026
5592,Equal College Access,Christian Castillo,2017.0,9,{},Assignment (4 Steps: 10 Annotations):\n1. High...,3017311
18372,Equal Access To College,Nahsiem Mccombs,2017.0,10,{},Assignment (4 Steps: 10 Annotations):\n1. High...,3016034
636,Penetration Assemblage 23 remix,Haujobb,2015.0,8,{},For a little piece of a second\nI have seen me...,1821933
27697,Penetration Converter remix,Haujobb,2015.0,11,{},For a little piece of a second\nI have seen me...,1812626
13159,Anon’s “We Are Anonymous” Haley Drozdowicz,#CA346S15,2015.0,3,"{""Andrew Zolides""}","GREETINGS, PEOPLE OF THE WORLD - WE ARE ANONYM...",721039
2120,Anon’s “We Are Anonymous” Roy Rosas,#CA346S15,2015.0,3,"{""Jeremy Morris""}","GREETINGS, PEOPLE OF THE WORLD - WE ARE ANONYM...",725679
24308,White Christmas,Rosie O'Donnell & Elton John,1999.0,111,"{""Rosie O\\'Donnell & Elton John""}",I'm dreaming of a white Christmas\nJust like t...,2095015
14776,White Christmas,Michael W. Smith,2014.0,91,{},I'm dreaming of a white Christmas\nJust like t...,1853129


Shared lyrics between train and test: 0


The empty training DataFrame and `Train: 0 rows with duplicated lyrics` confirm that no repeated lyric rows remain in train. The 24 displayed test rows are expected: duplicate removal deliberately does not modify the test set. `Shared lyrics between train and test: 0` confirms that the overlapping lyrics have been removed from training.

<h3 style="color: #597edb; font-weight: 400;">2.4 Imputation of missing values <a id="duplicate-audit"></a></h3>

<h4>2.4.1 Imputing missing years <a id="release-years"></a></h4>

The labeled data is split into training and validation sets using an 80/20 stratified split (`random_state=42`). We first count missing `year` values in the training, validation, and test sets. Missing years are then imputed using statistics calculated only from the original training split: each artist's median year is used when available, and the overall training median is the fallback for missing or unseen artists. Keeping the original training split as the reference ensures validation values do not contribute to the imputation statistics.

In [13]:
X_train_split, X_val, y_train_split, y_val = train_test_split(
    normalized_x_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

print(f"Training set: {len(X_train_split)} rows")
print(f"Validation set: {len(X_val)} rows")

Training set: 107746 rows
Validation set: 26937 rows


In [14]:
missing_years_summary = pd.DataFrame({
    'dataset': ['Training', 'Validation', 'Test'],
    'rows_with_missing_year': [
        X_train_split['year'].isna().sum(),
        X_val['year'].isna().sum(),
        X_test['year'].isna().sum(),
    ],
})

display(missing_years_summary)

,dataset,rows_with_missing_year
0,Training,312
1,Validation,76
2,Test,0


**Before imputation:** The table shows 312 missing years in the training split and 76 in validation. The test set has no missing years in this snapshot. This is the baseline used to check whether the imputation step filled the gaps.

In [15]:
from tools.preprocessing import imputing_years

year_imputation_reference = X_train_split.copy()
X_train_split = imputing_years(X_train_split, year_imputation_reference)
X_val = imputing_years(X_val, year_imputation_reference)

display(pd.DataFrame({
    "dataset": ["Training", "Validation"],
    "rows_with_missing_year": [
        X_train_split["year"].isna().sum(),
        X_val["year"].isna().sum(),
    ],
}))

,dataset,rows_with_missing_year
0,Training,0
1,Validation,0


**After imputation:** The output reports zero missing years in both training and validation, down from 312 and 76 respectively. Artist-level medians fill rows for artists represented with known years in training; the overall training median fills rows without a usable artist-specific median. The same original training reference is used for both splits, so validation data does not affect the imputation values.

<h4>2.4.2 Recovering missing titles <a id="missing-titles"></a></h4>

This step checks the training, validation, and test feature frames for missing song titles. For each missing title, it requests title metadata from the Genius API using the row's song ID, then fills the title when the lookup returns one. If a title cannot be retrieved, its value remains missing. The final table counts missing values in every column for each split. This updates the in-memory frames; rerun the appropriate save cell if these recovered titles should be written to disk.

In [16]:
missing_counts = X_train_split.isna().sum().rename("missing_values train")
display(missing_counts.to_frame())

,missing_values train
title,1
artist,0
year,0
views,0
features,0
lyrics,0
id,0


**Before recovery:** The training split has one missing `title`; the other displayed training columns have no missing values. The next cell attempts to retrieve missing titles in all three splits.

In [17]:
frames = {
    "Train": X_train_split,
    "Validation": X_val,
    "Test": X_test,
}

for split_name, data in frames.items():
    missing_title = data["title"].isna()
    missing_ids = data.loc[missing_title, "id"]

    if not missing_ids.empty:
        fetched = get_info_batch(missing_ids, title=True)
        if fetched is not None and "title" in fetched.columns:
            titles = data.loc[missing_title, "id"].map(fetched["title"])
            data.loc[missing_title, "title"] = titles.combine_first(
                data.loc[missing_title, "title"]
            )

missing_summary = pd.DataFrame({
    split_name: data.isna().sum()
    for split_name, data in frames.items()
})

display(missing_summary)

,Train,Validation,Test
title,0,0,0
artist,0,0,0
year,0,0,0
views,0,0,0
features,0,0,0
lyrics,0,0,0
id,0,0,0


**After recovery:** The table reports zero missing values in all listed columns for train, validation, and test. In this run, the missing title lookup succeeded, so no missing titles remain. If an API lookup fails or returns no title in a future run, the corresponding title will remain missing and appear in this summary.

In [18]:
clean_train = X_train_split.join(y_train_split.rename("tag"))
clean_val = X_val.join(y_val.rename("tag"))

if y_test["id"].duplicated().any():
    raise ValueError("y_test contains duplicate IDs.")

test_tags = X_test["id"].map(y_test.set_index("id")["tag"])
if test_tags.isna().any():
    raise ValueError("Some X_test IDs have no matching label in y_test.")

clean_test = X_test.copy()
clean_test["tag"] = test_tags

clean_train.to_csv(PROJECT_ROOT / "data" / "clean_train.csv", index=False)
clean_val.to_csv(PROJECT_ROOT / "data" / "clean_val.csv", index=False)
clean_test.to_csv(PROJECT_ROOT / "data" / "clean_test.csv", index=False)